# RandomSelector

Picks `n_features` columns uniformly at random. It ignores the data and the
target entirely.

It is the control. A method that does not beat random selection at the same
`k` contributed nothing, however good its score looks.

- **Strength:** an honest reference point, free to compute.
- **Reproducible:** the same `random_state` picks the same columns. The runner
  gives each split its own seed, so the control redraws per split.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
from sklearn.datasets import make_classification

from featsel.selectors import ANOVAFSelector, RandomSelector

X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## How often chance finds the informative features

Choosing k of p columns at random catches, on average, k x m / p of the m
informative ones.

In [2]:
k, p, m = 50, X.shape[1], len(truth)
hits = [len(truth & set(RandomSelector(n_features=k, random_state=s).fit(X, y)
                         .get_support(indices=True)))
        for s in range(1000)]
print(f'expected {k * m / p:.2f}, observed mean over 1000 seeds {np.mean(hits):.2f}')

expected 1.00, observed mean over 1000 seeds 1.04


## Against a real selector at the same k

In [3]:
anova_hits = len(truth & set(ANOVAFSelector(n_features=k).fit(X, y).get_support(indices=True)))
print(f'ANOVA F finds {anova_hits} of {m}; random finds {np.mean(hits):.1f} on average')

ANOVA F finds 5 of 10; random finds 1.0 on average


## Same seed, same columns; the target is ignored

In [4]:
a = RandomSelector(n_features=5, random_state=7).fit(X, y).get_support(indices=True)
b = RandomSelector(n_features=5, random_state=7).fit(X, rng.permutation(y)).get_support(indices=True)
print(a, b, 'identical:', np.array_equal(a, b))

[289 310 340 447 468] [289 310 340 447 468] identical: True
